In [ ]:
# Install required packages (run only once in your environment)
#%pip install pandas numpy matplotlib seaborn

In [ ]:
# Step 1: Data Understanding & Ingestion
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# consts
file_path = "../req/MediCare_Readmission_Intelligence.csv"
cleaned_file_path = "../temp/cleaned_dataSet.csv"

In [ ]:
# Load the dataset
df = pd.read_csv(file_path)

In [ ]:
# Basic inspection
print("Shape of dataset:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

In [ ]:
# Preview first 5 rows
print("\nSample rows:")
print(df.head())

In [ ]:
# Check target variable distribution
if 'readmission_flag' in df.columns:
    print("\nReadmission Flag Distribution:")
    print(df['readmission_flag'].value_counts())

# Summary statistics
print("\nSummary statistics:")
print(df.describe(include='all'))


# Key take aways from the above

Rows: 50500 rows
Columns: 31 
Target Variable: readmission_flag --> 32989 No and 17511 Yes (approx 35%)
Features: Mix of numerical (BP, sugar, cholesterol, etc), categorical (gender, insurance_type, smoking_status, discharge_destination) and administrative IDs.

Dataset is imbalanced

Several categorical features need normalization (insurance casing, discharge destinations)

Administrative IDs (legacy_record_id, billing_reference_number, care_cluster_id, administrative_batch_id) are likely non-predictive noise and should be dropped later.



In [ ]:
# Step 2: Data Cleaning & Preprocessing

# 1. Remove duplicate rows
print("Rows before duplicate removal:", df.shape[0])
df = df.drop_duplicates()
print("Rows after duplicate removal:", df.shape[0])

In [ ]:
# 2. Drop irrelevant columns (if any)
cols_to_drop = [
    "legacy_record_id", 
    "billing_reference_number",
    "care_cluster_id",
    "administrative_batch_id",
    "patient_education_score" #remove this column as at this point of time it is not making any sense
]

df = df.drop(columns = cols_to_drop, errors='ignore')

In [ ]:
# 2. Standardize categorical values 

cat_cols = [
    "gender", "smoking_status", "insurance_type", "discharge_destination"
]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

print(df.head())


In [ ]:
# 3. Data type corrections
#['patient_id', 'age', 'gender', 'bmi', 'smoking_status', 'diabetes_flag', 'hypertension_flag', 'heart_disease_flag', 'chronic_conditions_count', 'previous_admissions_12m', 'length_of_stay_days', 'icu_admission_flag', 'emergency_admission_flag', 'number_of_procedures', 'blood_glucose', 'cholesterol_level', 'hemoglobin', 'creatinine', 'medications_count', 'high_risk_medication_flag', 'medication_changes_during_stay', 'followup_scheduled_flag', 'discharge_destination', 'patient_education_score', 'insurance_type', 'treatment_cost', 'legacy_record_id', 'billing_reference_number', 'care_cluster_id', 'administrative_batch_id', 'readmission_flag']


numeric_cols = [
    "age", 
    "bmi", 
    "chronic_conditions_count", 
    "previous_admissions_12m", 
    "length_of_stay_days", 
    "number_of_procedures", 
    "blood_glucose", 
    "cholesterol_level", 
    "hemoglobin", 
    "creatinine", 
    "medications_count",
    "treatment_cost"
]

binary_cols = [
    "diabetes_flag",
    "hypertension_flag",
    "heart_disease_flag",
    "icu_admission_flag",
    "emergency_admission_flag",
    "high_risk_medication_flag",
    "followup_scheduled_flag",
    "readmission_flag"
]

# Convert numeric columns to appropriate data types
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Convert binary columns to numeric
df[binary_cols] = df[binary_cols].apply(pd.to_numeric, errors='coerce')

print(df.head())

In [ ]:
# 4. Missing data handling

# % missing check
missing_pct = df.isnull().mean() * 100
print("\nMissing data percentage per column:\n", missing_pct[missing_pct > 0])

# Numerical columns: Impute missing values with median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col].fillna(median_value, inplace=True)
        print(f"Imputed missing values in {col} with median: {median_value}")

# Categorical columns: Impute missing values with mode
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()[0]
        df[col].fillna(mode_value, inplace=True)
        print(f"Imputed missing values in {col} with mode: {mode_value}")

# Verify no missing values remain
print("\nMissing data after imputation:\n", df.isnull().sum())

print("\nFinal dataset shape after cleaning:", df.shape)


In [ ]:
# 5. Outlier detection and handling - Winsorization

def cap_outliers(series, lower = 0.01, upper = 0.99):
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower=lower_bound, upper=upper_bound)

for col in numeric_cols:
    df[col] = cap_outliers(df[col])

print("\nFinal dataset shape after cleaning:", df.shape)

In [ ]:
# 6. Establishing rules for data validation

#Idea is to remove impossible values, 
df = df[(df['age'] >= 0)]  # Age cannot be negative
df = df[(df['bmi'] >= 0)] # BMI should be in a reasonable range (0-10 for this dataset)
df = df[df['length_of_stay_days'] >= 0]  # Length of stay cannot be negative
df = df[df['blood_glucose'] >= 0]  # Blood glucose cannot be negative
df = df[df['cholesterol_level'] >= 0]  # Cholesterol level cannot be negative

# Capping extreme realistic values
df["age"] = np.clip(df["age"], 0, 120)  # Age should be between 0 and 120
df["bmi"] = np.clip(df["bmi"], 10, 60)  # BMI should be between 10 and 60

print("\nFinal dataset shape after cleaning:", df.shape)

In [ ]:
# Final Sanity Check
print("\nFinal dataset shape after cleaning:", df.shape)


In [ ]:
# Save the cleaned dataset for further analysis
df.to_csv(cleaned_file_path, index=False)